## **Data Processing**
### Note, run ThimkersRemoteWork.ipynb first before running this
### You must also use the exact same kernel to keep the variables
---

In [1]:
# %pip install scikit_posthocs
# %pip install scikit-learn
# %pip install plotly

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

from scipy import stats
from scipy.stats import mannwhitneyu

from scipy.stats import pearsonr, spearmanr, levene, f_oneway, shapiro, kruskal, chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from matplotlib.colors import ListedColormap

In [2]:

%store -r
print("Variables restored successfully.")


Variables restored successfully.


In [3]:
print("Current Variables")
print(f"target                      : {target.shape}")
print(f"current_profession_encoded  : {current_profession_encoded.shape}")
print(f"age_group                   : {age_group.shape}")
print(f"education_level             : {education_level.shape}")
print(f"employment_status           : {employment_status.shape}")
print(f"dev_type_encoded            : {dev_type_encoded.shape}")
print(f"work_years (non-null)       : {work_years.notna().sum()}")
print(f"learn_years (non-null)      : {learn_years.notna().sum()}")
print(f"org_size_ordinal            : {org_size_ordinal.shape}")
print(f"work_tool_count             : {work_tool_count.shape}")
print(f"personal_tool_count         : {personal_tool_count.shape}")
print(f"geographic_regions_encoded  : {geographic_regions_encoded.shape}")
print(f"language_features           : {language_features.shape}")
print(f"database_features           : {database_features.shape}")
print(f"platform_features           : {platform_features.shape}")
print(f"webframe_features           : {webframe_features.shape}")
print(f"devenv_features             : {devenv_features.shape}")
print(f"collab_features             : {collab_features.shape}")
print(f"aimodel_features            : {aimodel_features.shape}")
print(f"ai_industry_use             : {ai_industry_use.shape}")
print(f"ai_learn_how                : {ai_learn_how.shape}")
print(f"learncodeai_encoded         : {learncodeai_encoded.shape}")
print(f"aiselect_encoded            : {aiselect_encoded.shape}")
print(f"aiagents_encoded            : {aiagents_encoded.shape}")
print(f"aiagentchange_encoded       : {aiagentchange_encoded.shape}")
print(f"ai_technical_use            : {ai_technical_use.shape}")
print(f"ai_knowledge                : {ai_knowledge.shape}")
print(f"ai_orchestration            : {ai_orchestration.shape}")
print(f"ai_observe_secure           : {ai_observe_secure.shape}")
print(f"ai_external                 : {ai_external.shape}")

Current Variables
target                      : (49191,)
current_profession_encoded  : (49191, 4)
age_group                   : (49191, 6)
education_level             : (49191, 8)
employment_status           : (49191, 5)
dev_type_encoded            : (49191, 21)
work_years (non-null)       : 42893
learn_years (non-null)      : 43042
org_size_ordinal            : (49191,)
work_tool_count             : (49191,)
personal_tool_count         : (49191,)
geographic_regions_encoded  : (49191, 19)
language_features           : (49191, 42)
database_features           : (49191, 30)
platform_features           : (49191, 42)
webframe_features           : (49191, 28)
devenv_features             : (49191, 27)
collab_features             : (49191, 25)
aimodel_features            : (49191, 17)
ai_industry_use             : (49191, 10)
ai_learn_how                : (49191, 13)
learncodeai_encoded         : (49191, 2)
aiselect_encoded            : (49191, 4)
aiagents_encoded            : (49191, 4)
aiage

## **All Features and Train/Test Split**

In [4]:
X = pd.concat([
    current_profession_encoded,
    age_group,
    education_level,
    employment_status,
    dev_type_encoded,
    geographic_regions_encoded,
    pd.DataFrame({'org_size': org_size_ordinal}),
    pd.DataFrame({'work_exp': work_years}),
    pd.DataFrame({'years_code': learn_years}),
    pd.DataFrame({'work_tools': work_tool_count}),
    pd.DataFrame({'personal_tools': personal_tool_count}),
    language_features,
    database_features,
    platform_features,
    webframe_features,
    devenv_features,
    collab_features,
    aimodel_features,
    ai_industry_use,
    ai_learn_how,
    learncodeai_encoded,
    aiselect_encoded,
    aiagents_encoded,
    aiagentchange_encoded,
    ai_technical_use,
    ai_knowledge,
    ai_orchestration,
    ai_observe_secure,
    ai_external,
], axis=1)

y = target

# Remove NaN and set median data
X_clean = X.copy()
X_clean = X_clean.fillna(0)

# 
for col in ['work_exp', 'years_code', 'work_tools', 'personal_tools', 'org_size']:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

print(f"Full matrix: {X_clean.shape}")
print(f"Target: {y.shape}")
print(f"Target Ratio: {y.value_counts().to_dict()}")

Full matrix: (49191, 395)
Target: (49191,)
Target Ratio: {0: 34016, 1: 15175}


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize Features
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Oversampling with SMOTE
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

print(f"Train size: {X_train_scaled.shape}")
print(f"Test size: {X_test_scaled.shape}")
print(f"Train class balance: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test  class balance: {pd.Series(y_test).value_counts().to_dict()}")

Train size: (54424, 395)
Test size: (9839, 395)
Train class balance: {1: 27212, 0: 27212}
Test  class balance: {0: 6804, 1: 3035}


## **K-Nearest Neighbors (KNN)**

In [12]:
k_range = range(1, 31)
train_errors = []
test_errors  = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_errors.append(1 - knn.score(X_train_scaled, y_train))
    test_errors.append(1 - knn.score(X_test_scaled, y_test))
    print(f"K={k:<3}  Train Error: {train_errors[-1]:.4f}  Test Error: {test_errors[-1]:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(k_range), y=train_errors,
    mode='lines+markers', name='Train Error',
    line=dict(color='royalblue')
))
fig.add_trace(go.Scatter(
    x=list(k_range), y=test_errors,
    mode='lines+markers', name='Test Error',
    line=dict(color='tomato')
))

# Get the best
best_k = test_errors.index(min(test_errors)) + 1
fig.add_vline(x=best_k, line_dash='dash', line_color='green',
              annotation_text=f'Best K={best_k}', annotation_position='top right')

fig.update_layout(
    title='KNN - Error Rate vs Number of Neighbors (K)',
    xaxis_title='K (n_neighbors)',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

print(f"Best K by lowest test error : K = {best_k}")
print(f"Train error : {train_errors[best_k - 1]:.4f}")
print(f"Test error  : {test_errors[best_k - 1]:.4f}")


K=1    Train Error: 0.0003  Test Error: 0.4004
K=2    Train Error: 0.0205  Test Error: 0.3773
K=3    Train Error: 0.1862  Test Error: 0.4307
K=4    Train Error: 0.1748  Test Error: 0.4082
K=5    Train Error: 0.2458  Test Error: 0.4432
K=6    Train Error: 0.2346  Test Error: 0.4234
K=7    Train Error: 0.2763  Test Error: 0.4477
K=8    Train Error: 0.2670  Test Error: 0.4301
K=9    Train Error: 0.2941  Test Error: 0.4492
K=10   Train Error: 0.2863  Test Error: 0.4360
K=11   Train Error: 0.3053  Test Error: 0.4533
K=12   Train Error: 0.2990  Test Error: 0.4388
K=13   Train Error: 0.3134  Test Error: 0.4568
K=14   Train Error: 0.3073  Test Error: 0.4477
K=15   Train Error: 0.3193  Test Error: 0.4600
K=16   Train Error: 0.3152  Test Error: 0.4513
K=17   Train Error: 0.3246  Test Error: 0.4608
K=18   Train Error: 0.3206  Test Error: 0.4539
K=19   Train Error: 0.3293  Test Error: 0.4635
K=20   Train Error: 0.3258  Test Error: 0.4561
K=21   Train Error: 0.3326  Test Error: 0.4635
K=22   Train 

Best K by lowest test error : K = 2
Train error : 0.0205
Test error  : 0.3773


In [13]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)
knn_predictions = knn_best.predict(X_test_scaled)

cm_knn = confusion_matrix(y_test, knn_predictions)
acc_knn = accuracy_score(y_test, knn_predictions)
report_knn = classification_report(y_test, knn_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_knn, fp_knn, fn_knn, tp_knn = cm_knn.ravel()

print(f"KNN (K={best_k}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 63)
print(f"{'Accuracy':<25} {acc_knn:>12.4f}")
print(f"{'Precision':<25} {report_knn['Non-Remote']['precision']:>12.4f} {report_knn['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_knn['Non-Remote']['recall']:>12.4f} {report_knn['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_knn['Non-Remote']['f1-score']:>12.4f} {report_knn['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_knn['Non-Remote']['support']:>12} {report_knn['Remote']['support']:>12}")
print(f"{'Macro Avg F1':<25} {report_knn['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_knn['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_knn,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_knn), str(fp_knn)], [str(fn_knn), str(tp_knn)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'KNN (K={best_k}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


KNN (K=2) Results
Metric                      Non-Remote       Remote
---------------------------------------------------------------
Accuracy                        0.6227
Precision                       0.7764       0.4203
Recall                          0.6383       0.5878
F1 Score                        0.7006       0.4901
Support                         6804.0       3035.0
Macro Avg F1                    0.5954
Weighted Avg F1                 0.6357


## **Logistic Regression**

In [14]:
C_range = [0.001, 0.01, 0.1, 1, 10, 100]
C_strings = [str(c) for c in C_range]

# Not all penalty and solver combinations are valid
hyperparam_combos = [
    ('l2', 'lbfgs', {}),
    ('l2', 'liblinear', {}),
    ('l1', 'liblinear', {}),
    ('l1', 'saga', {}),
    ('elasticnet', 'saga', {'l1_ratio': 0.5}),
    (None, 'lbfgs', {}),
]

lr_results = {}

for i, (penalty, solver, extra) in enumerate(hyperparam_combos, 1):
    label = f"{penalty or 'none'}/{solver}"
    print(f"[{i}/{len(hyperparam_combos)}] Fitting: {label}")
    train_errors, test_errors = [], []
    for C in C_range:
        model = LogisticRegression(penalty=penalty, solver=solver, C=C, max_iter=1000, random_state=42, **extra)
        model.fit(X_train_scaled, y_train)
        train_errors.append(1 - model.score(X_train_scaled, y_train))
        test_errors.append(1 - model.score(X_test_scaled, y_test))
        print(f"  C={str(C):<8}  Train Error: {train_errors[-1]:.4f}  Test Error: {test_errors[-1]:.4f}")
    lr_results[label] = {'train': train_errors, 'test': test_errors}

# Plot test error for all combos
fig = go.Figure()
for combo, errors in lr_results.items():
    fig.add_trace(go.Scatter(
        x=C_strings, y=errors['test'],
        mode='lines+markers', name=combo
    ))

fig.update_layout(
    title='Logistic Regression - Test Error vs C by Penalty/Solver',
    xaxis_title='C (Inverse Regularization Strength)',
    yaxis_title='Test Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

# Find best overall combo and C
best_lr_label, best_C, best_C_idx, best_err = None, None, None, 1.0

for combo, errors in lr_results.items():
    index = errors['test'].index(min(errors['test']))
    if errors['test'][index] < best_err:
        best_err = errors['test'][index]
        best_lr_label = combo
        best_C_idx = index
        best_C = C_range[index]

best_penalty, best_solver = best_lr_label.split('/')
best_penalty = None if best_penalty == 'none' else best_penalty
best_extra = {'l1_ratio': 0.5} if best_penalty == 'elasticnet' else {}

print(f"Best combo: {best_lr_label}")
print(f"Best C: {best_C}")
print(f"Train error: {lr_results[best_lr_label]['train'][best_C_idx]:.4f}")
print(f"Test error: {best_err:.4f}")


[1/6] Fitting: l2/lbfgs
  C=0.001     Train Error: 0.2918  Test Error: 0.3347
  C=0.01      Train Error: 0.2908  Test Error: 0.3389
  C=0.1       Train Error: 0.2900  Test Error: 0.3416
  C=1         Train Error: 0.2898  Test Error: 0.3419
  C=10        Train Error: 0.2897  Test Error: 0.3422
  C=100       Train Error: 0.2897  Test Error: 0.3420
[2/6] Fitting: l2/liblinear
  C=0.001     Train Error: 0.2917  Test Error: 0.3377
  C=0.01      Train Error: 0.2905  Test Error: 0.3398
  C=0.1       Train Error: 0.2899  Test Error: 0.3411
  C=1         Train Error: 0.2899  Test Error: 0.3418
  C=10        Train Error: 0.2899  Test Error: 0.3418
  C=100       Train Error: 0.2899  Test Error: 0.3418
[3/6] Fitting: l1/liblinear
  C=0.001     Train Error: 0.3163  Test Error: 0.3727
  C=0.01      Train Error: 0.2951  Test Error: 0.3412
  C=0.1       Train Error: 0.2909  Test Error: 0.3416
  C=1         Train Error: 0.2900  Test Error: 0.3417
  C=10        Train Error: 0.2899  Test Error: 0.3418
  

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning:

The max_iter was reached which means the coef_ did not converge



  C=0.1       Train Error: 0.2910  Test Error: 0.3416


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning:

The max_iter was reached which means the coef_ did not converge



  C=1         Train Error: 0.2900  Test Error: 0.3416


...

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning:

Setting penalty=None will ignore the C and l1_ratio parameters



  C=10        Train Error: 0.2897  Test Error: 0.3422


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning:

Setting penalty=None will ignore the C and l1_ratio parameters



  C=100       Train Error: 0.2897  Test Error: 0.3422


Best combo: l2/lbfgs
Best C: 0.001
Train error: 0.2918
Test error: 0.3347


In [15]:
lr_best = LogisticRegression(penalty=best_penalty, solver=best_solver, C=best_C,
    max_iter=1000, random_state=42, **best_extra
)

lr_best.fit(X_train_scaled, y_train)
lr_predictions = lr_best.predict(X_test_scaled)

cm_lr = confusion_matrix(y_test, lr_predictions)
acc_lr = accuracy_score(y_test, lr_predictions)
report_lr = classification_report(y_test, lr_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()

print(f"Logistic Regression ({best_lr_label}, C={best_C}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_lr['Non-Remote']['precision']:>12.4f} {report_lr['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_lr['Non-Remote']['recall']:>12.4f} {report_lr['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_lr['Non-Remote']['f1-score']:>12.4f} {report_lr['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_lr['Non-Remote']['support']:>12} {report_lr['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_lr:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_lr['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_lr['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_lr,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_lr), str(fp_lr)], [str(fn_lr), str(tp_lr)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Logistic Regression ({best_lr_label}, C={best_C}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Logistic Regression (l2/lbfgs, C=0.001) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.8400       0.4724
Recall                          0.6374       0.7278
F1 Score                        0.7248       0.5729
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.6653
Macro Avg F1                    0.6489
Weighted Avg F1                 0.6780


In [16]:

# Getting top variables with .coef
feature_names = X_clean.columns.tolist()
coefs = lr_best.coef_[0] 

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefs
})
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=False).head(20)
coef_df = coef_df.sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'royalblue' for c in coef_df['Coefficient']]

fig = go.Figure(go.Bar(
    x=coef_df['Coefficient'],
    y=coef_df['Feature'],
    orientation='h',
    marker_color=colors
))
fig.update_layout(
    title=f'Logistic Regression ({best_lr_label}, C={best_C}) - Top 20 Feature Coefficients',
    xaxis_title='Coefficient Value',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print("Top 20 features by absolute coefficient:")
print(f"{'Rank':<6} {'Feature':<40} {'Coefficient':>12}")
print("-" * 60)
for rank, (index, row) in enumerate(coef_df.sort_values('Abs', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<40} {row['Coefficient']:>12.4f}")


Top 20 features by absolute coefficient:
Rank   Feature                                   Coefficient
------------------------------------------------------------
1      devtype_full-stack developer                   0.3447
2      learncodeai_no                                 0.3177
3      employment_employed                            0.3164
4      org_size                                       0.2980
5      devtype_backend developer                      0.2969
6      learncodeai_yes                                0.2825
7      employment_unemployed                         -0.2245
8      devtype_frontend developer                     0.1911
9      years_code                                     0.1726
10     region_northern_america                        0.1698
11     devtype_mobile developer                       0.1649
12     region_eastern_europe                          0.1532
13     devtype_software architect                     0.1467
14     work_exp                             

## **Support Vector Machine (SVM)**

In [ ]:
#SVM Hyperparameter Tuning
C_range = [0.001, 0.01, 0.1, 1, 10, 100]
kernel_options = ['linear', 'rbf', 'poly']
svm_results = {}
for kernel in kernel_options:
    train_errors, test_errors = [], []
    print(f"Testing SVM with kernel: {kernel}")
    for C in C_range:
        svm = SVC(kernel=kernel, C=C, random_state=42)
        svm.fit(X_train_scaled, y_train)
        train_errors.append(1 - svm.score(X_train_scaled, y_train))
        test_errors.append(1 - svm.score(X_test_scaled, y_test))
        print(f"  C={str(C):<8}  Train Error: {train_errors[-1]:.4f}  Test Error: {test_errors[-1]:.4f}")
    svm_results[kernel] = {'train': train_errors, 'test': test_errors}
# Plot SVM results
fig = go.Figure()
for kernel in kernel_options:
    fig.add_trace(go.Scatter(x=C_range, y=svm_results[kernel]['test'], mode='lines+markers', name=f'{kernel} Test'))
    fig.add_trace(go.Scatter(x=C_range, y=svm_results[kernel]['train'], mode='lines+markers', name=f'{kernel} Train'))
fig.update_layout(
    title='SVM - Error Rate vs C by Kernel',
    xaxis_title='C (Inverse Regularization Strength)',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=500
)
fig.show()
# Find best SVM combo
best_svm_kernel, best_svm_C, best_svm_C_idx, best_svm_err = None, None, None, 1.0
for kernel in kernel_options:
    index = svm_results[kernel]['test'].index(min(svm_results[kernel]['test']))
    if svm_results[kernel]['test'][index] < best_svm_err:
        best_svm_err = svm_results[kernel]['test'][index]
        best_svm_kernel = kernel
        best_svm_C_idx = index
        best_svm_C = C_range[index]
print(f"Best SVM combo: Kernel={best_svm_kernel}, C={best_svm_C}")
print(f"Train error: {svm_results[best_svm_kernel]['train'][best_svm_C_idx]:.4f}")
print(f"Test error: {best_svm_err:.4f}")
svm_best = SVC(kernel=best_svm_kernel, C=best_svm_C, random_state=42)
svm_best.fit(X_train_scaled, y_train)
svm_predictions = svm_best.predict(X_test_scaled)
cm_svm = confusion_matrix(y_test, svm_predictions)
acc_svm = accuracy_score(y_test, svm_predictions)
report_svm = classification_report(y_test, svm_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_svm, fp_svm, fn_svm, tp_svm = cm_svm.ravel()
print(f"SVM (Kernel={best_svm_kernel}, C={best_svm_C}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_svm['Non-Remote']['precision']:>12.4f} {report_svm['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_svm['Non-Remote']['recall']:>12.4f} {report_svm['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_svm['Non-Remote']['f1-score']:>12.4f} {report_svm['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_svm['Non-Remote']['support']:>12} {report_svm['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_svm:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_svm['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_svm['weighted avg']['f1-score']:>12.4f}")
fig = go.Figure(data=go.Heatmap(
    z=cm_svm,
    x=['Non-Remote', 'Remote'],
    y=['Non-Remote', 'Remote'],
    colorscale='Blues',
    text=cm_svm,
    texttemplate="%{text}",
    hoverongaps=False
))
fig.update_layout(
    title='SVM Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    template='plotly_white'
)
fig.show()

Testing SVM with kernel: linear
  C=0.001     Train Error: 0.2893  Test Error: 0.3515
  C=0.01      Train Error: 0.2910  Test Error: 0.3533


## **Naive Bayes**

In [9]:
# Naive Bayes
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
nb_predictions = nb.predict(X_test_scaled)
cm_nb = confusion_matrix(y_test, nb_predictions)
acc_nb = accuracy_score(y_test, nb_predictions)
report_nb = classification_report(y_test, nb_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_nb, fp_nb, fn_nb, tp_nb = cm_nb.ravel()
print(f"Naive Bayes Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_nb['Non-Remote']['precision']:>12.4f} {report_nb['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_nb['Non-Remote']['recall']:>12.4f} {report_nb['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_nb['Non-Remote']['f1-score']:>12.4f} {report_nb['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_nb['Non-Remote']['support']:>12} {report_nb['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_nb:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_nb['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_nb['weighted avg']['f1-score']:>12.4f}")
fig = go.Figure(data=go.Heatmap(
    z=cm_nb,
    x=['Non-Remote', 'Remote'],
    y=['Non-Remote', 'Remote'],
    colorscale='Blues',
    text=cm_nb,
    texttemplate="%{text}",
    hoverongaps=False
))
fig.update_layout(
    title="Confusion Matrix - Naive Bayes",
    xaxis_title="Predicted",
    yaxis_title="Actual"
)
fig.show()

Naive Bayes Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.8162       0.3999
Recall                          0.4993       0.7479
F1 Score                        0.6196       0.5211
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.5760
Macro Avg F1                    0.5703
Weighted Avg F1                 0.5892


## **Random Forest**

In [17]:
n_estimators_range  = [10, 50, 100, 200, 300, 500]
n_estimators_strings = [str(n) for n in n_estimators_range]

max_depth_range = [None, 5, 10, 20]
bootstrap_range = [True, False]

rf_combos = [
    (depth, option) for depth in max_depth_range for option in bootstrap_range
]

rf_results = {}

for i, (depth, boot) in enumerate(rf_combos, 1):
    label = f"depth={'None' if depth is None else depth}/bootstrap={boot}"
    print(f"[{i}/{len(rf_combos)}] Fitting: {label}")
    rf_train_errors = []
    rf_test_errors  = []
    for n in n_estimators_range:
        rf = RandomForestClassifier(n_estimators=n, max_depth=depth, bootstrap=boot, random_state=42, 
                                    n_jobs=-1)
        rf.fit(X_train_scaled, y_train)
        rf_train_errors.append(1 - rf.score(X_train_scaled, y_train))
        rf_test_errors.append(1 - rf.score(X_test_scaled, y_test))
        print(f"n_estimators={str(n):<6}  Train Error: {rf_train_errors[-1]:.4f}  Test Error: {rf_test_errors[-1]:.4f}")
    rf_results[label] = {'train': rf_train_errors, 'test': rf_test_errors}

fig = go.Figure()
for combo, errors in rf_results.items():
    fig.add_trace(go.Scatter(
        x=n_estimators_strings, y=errors['test'],
        mode='lines+markers', name=combo
    ))

fig.update_layout(
    title='Random Forest - Test Error vs n_estimators by max_depth/bootstrap',
    xaxis_title='n_estimators',
    yaxis_title='Test Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

best_rf_label, best_n, best_n_idx, best_rf_err = None, None, None, 1.0

for combo, errors in rf_results.items():
    idx = errors['test'].index(min(errors['test']))
    if errors['test'][idx] < best_rf_err:
        best_rf_err    = errors['test'][idx]
        best_rf_label  = combo
        best_n_idx     = idx
        best_n         = n_estimators_range[idx]

depth_choice, boot_choice = best_rf_label.split('/')
best_depth = None if 'None' in depth_choice else int(depth_choice.split('=')[1])
best_bootstrap = boot_choice.split('=')[1] == 'True'

print(f"Best combo: {best_rf_label}")
print(f"Best n_estimators: {best_n}")
print(f"Train error: {rf_results[best_rf_label]['train'][best_n_idx]:.4f}")
print(f"Test error: {best_rf_err:.4f}")


[1/8] Fitting: depth=None/bootstrap=True
n_estimators=10      Train Error: 0.0070  Test Error: 0.2841
n_estimators=50      Train Error: 0.0003  Test Error: 0.2658
n_estimators=100     Train Error: 0.0003  Test Error: 0.2598
n_estimators=200     Train Error: 0.0003  Test Error: 0.2530
n_estimators=300     Train Error: 0.0003  Test Error: 0.2509
n_estimators=500     Train Error: 0.0003  Test Error: 0.2493
[2/8] Fitting: depth=None/bootstrap=False
n_estimators=10      Train Error: 0.0003  Test Error: 0.2791
n_estimators=50      Train Error: 0.0003  Test Error: 0.2599
n_estimators=100     Train Error: 0.0003  Test Error: 0.2528
n_estimators=200     Train Error: 0.0003  Test Error: 0.2501
n_estimators=300     Train Error: 0.0003  Test Error: 0.2521
n_estimators=500     Train Error: 0.0003  Test Error: 0.2507
[3/8] Fitting: depth=5/bootstrap=True
n_estimators=10      Train Error: 0.2472  Test Error: 0.3347
n_estimators=50      Train Error: 0.2528  Test Error: 0.3256
n_estimators=100     Trai

Best combo: depth=None/bootstrap=True
Best n_estimators: 500
Train error: 0.0003
Test error: 0.2493


In [18]:
rf_best = RandomForestClassifier(n_estimators=best_n, max_depth=best_depth,
                                 bootstrap=best_bootstrap, random_state=42, n_jobs=-1)
rf_best.fit(X_train_scaled, y_train)
rf_predictions = rf_best.predict(X_test_scaled)

cm_rf = confusion_matrix(y_test, rf_predictions)
acc_rf = accuracy_score(y_test, rf_predictions)
report_rf = classification_report(y_test, rf_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

print(f"Random Forest ({best_rf_label}, n={best_n}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_rf['Non-Remote']['precision']:>12.4f} {report_rf['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_rf['Non-Remote']['recall']:>12.4f} {report_rf['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_rf['Non-Remote']['f1-score']:>12.4f} {report_rf['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_rf['Non-Remote']['support']:>12} {report_rf['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_rf:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_rf['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_rf['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_rf,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_rf), str(fp_rf)], [str(fn_rf), str(tp_rf)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Random Forest ({best_rf_label}, n={best_n}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Random Forest (depth=None/bootstrap=True, n=500) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.7873       0.6284
Recall                          0.8762       0.4692
F1 Score                        0.8294       0.5373
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.7507
Macro Avg F1                    0.6833
Weighted Avg F1                 0.7393


In [19]:
# Feature Importance
feature_names = X_clean.columns.tolist()
importances = rf_best.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})
importance_df = importance_df.sort_values('Importance', ascending=False).head(20)
importance_df = importance_df.sort_values('Importance')

fig = go.Figure(go.Bar(
    x=importance_df['Importance'],
    y=importance_df['Feature'],
    orientation='h',
    marker_color='royalblue'
))
fig.update_layout(
    title=f'Random Forest ({best_rf_label}, n={best_n}) - Top 20 Feature Importances',
    xaxis_title='Importance Score',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print("Top 20 features by importance:")
print(f"{'Rank':<6} {'Feature':<40} {'Importance':>12}")
print("-" * 60)
for rank, (index, row) in enumerate(importance_df.sort_values('Importance', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<40} {row['Importance']:>12.4f}")


Top 20 features by importance:
Rank   Feature                                    Importance
------------------------------------------------------------
1      org_size                                       0.1201
2      years_code                                     0.0548
3      work_exp                                       0.0474
4      employment_employed                            0.0296
5      work_tools                                     0.0175
6      personal_tools                                 0.0145
7      profession_professional dev                    0.0136
8      ai_learn_how_ai_codegen_tools_or_ai_enabled_apps       0.0121
9      region_northern_america                        0.0112
10     collab_jira                                    0.0109
11     age_group_35-44 years old                      0.0102
12     ed_level_bachelor                              0.0097
13     lang_python                                    0.0095
14     platform_amazon_web_services_aws       

## **Neural Network**

In [20]:

# NOTE: The full hyperparameter permutations become super slow and got stuck. 
# It didn't finish even after 9 hours. So the solver will only be Adam
# and the max iterations will be the default 200. Bro it got stuck at 83/144 after 800 mins ;-;

architectures = [
    (64,),
    (128,),
    (64, 64),
    (128, 64),
    (128, 128),
    (256, 128, 64),
]
activations = ['relu', 'tanh']
alpha_range = [0.0001, 0.001, 0.01]

mlp_combos = [
    (arch, act, alpha)
    for arch in architectures
    for act in activations
    for alpha in alpha_range
]

print(f"Total MLP combos: {len(mlp_combos)}")

mlp_results = {}

for i, (arch, act, alpha) in enumerate(mlp_combos, 1):
    label = f"{arch}/{act}/Regularization={alpha}"
    print(f"[{i}/{len(mlp_combos)}] Fitting: {label}")
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation=act,
        solver='adam',
        alpha=alpha,
        random_state=42,
        early_stopping=False
    )
    mlp.fit(X_train_scaled, y_train)
    train_err = 1 - mlp.score(X_train_scaled, y_train)
    test_err  = 1 - mlp.score(X_test_scaled, y_test)
    print(f"  Train Error: {train_err:.4f}  Test Error: {test_err:.4f}")
    mlp_results[label] = {'train': train_err, 'test': test_err}

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(mlp_results.keys()),
    y=[v['test'] for v in mlp_results.values()],
    name='Test Error',
    marker_color='tomato'
))
fig.add_trace(go.Bar(
    x=list(mlp_results.keys()),
    y=[v['train'] for v in mlp_results.values()],
    name='Train Error',
    marker_color='royalblue'
))
fig.update_layout(
    title='Neural Network (MLP, Adam) - Train/Test Error by Arch/Activation/Alpha',
    xaxis_title='Combo (Arch/Activation/Alpha)',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=600,
    barmode='group',
    xaxis_tickangle=-45,
    legend=dict(font=dict(size=9))
)
fig.show()

best_mlp_label = min(mlp_results, key=lambda k: mlp_results[k]['test'])
best_mlp_err   = mlp_results[best_mlp_label]['test']

label_parts = best_mlp_label.split('/')
arch_str = label_parts[0]
best_act = label_parts[1]
best_regularizer = float(label_parts[2].replace('Regularization=', ''))
best_arch = tuple(int(x) for x in arch_str.strip('()').split(',') if x.strip())
best_solver = 'adam'

print(f"\nBest combo    : {best_mlp_label}")
print(f"Architecture  : {best_arch}")
print(f"Activation    : {best_act}")
print(f"Alpha (L2)    : {best_regularizer}")
print(f"Solver        : {best_solver}")
print(f"Train error   : {mlp_results[best_mlp_label]['train']:.4f}")
print(f"Test error    : {best_mlp_err:.4f}")


Total MLP combos: 36
[1/36] Fitting: (64,)/relu/Regularization=0.0001


  Train Error: 0.0306  Test Error: 0.3232
[2/36] Fitting: (64,)/relu/Regularization=0.001
  Train Error: 0.0237  Test Error: 0.3131
[3/36] Fitting: (64,)/relu/Regularization=0.01
  Train Error: 0.0275  Test Error: 0.3171
[4/36] Fitting: (64,)/tanh/Regularization=0.0001


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.



  Train Error: 0.0150  Test Error: 0.3382
[5/36] Fitting: (64,)/tanh/Regularization=0.001
  Train Error: 0.0232  Test Error: 0.3430
[6/36] Fitting: (64,)/tanh/Regularization=0.01
  Train Error: 0.0205  Test Error: 0.3347
[7/36] Fitting: (128,)/relu/Regularization=0.0001
  Train Error: 0.0222  Test Error: 0.3015
[8/36] Fitting: (128,)/relu/Regularization=0.001
  Train Error: 0.0433  Test Error: 0.2973
[9/36] Fitting: (128,)/relu/Regularization=0.01
  Train Error: 0.0281  Test Error: 0.3069
[10/36] Fitting: (128,)/tanh/Regularization=0.0001
  Train Error: 0.0114  Test Error: 0.3283
[11/36] Fitting: (128,)/tanh/Regularization=0.001


C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.



  Train Error: 0.0106  Test Error: 0.3368
[12/36] Fitting: (128,)/tanh/Regularization=0.01
  Train Error: 0.0220  Test Error: 0.3265
[13/36] Fitting: (64, 64)/relu/Regularization=0.0001
  Train Error: 0.0168  Test Error: 0.3108
[14/36] Fitting: (64, 64)/relu/Regularization=0.001
  Train Error: 0.0198  Test Error: 0.3173
[15/36] Fitting: (64, 64)/relu/Regularization=0.01
  Train Error: 0.0200  Test Error: 0.3084
[16/36] Fitting: (64, 64)/tanh/Regularization=0.0001
  Train Error: 0.0101  Test Error: 0.3345
[17/36] Fitting: (64, 64)/tanh/Regularization=0.001
  Train Error: 0.0082  Test Error: 0.3309
[18/36] Fitting: (64, 64)/tanh/Regularization=0.01
  Train Error: 0.0166  Test Error: 0.3340
[19/36] Fitting: (128, 64)/relu/Regularization=0.0001
  Train Error: 0.0115  Test Error: 0.3041
[20/36] Fitting: (128, 64)/relu/Regularization=0.001
  Train Error: 0.0087  Test Error: 0.3122
[21/36] Fitting: (128, 64)/relu/Regularization=0.01
  Train Error: 0.0137  Test Error: 0.3006
[22/36] Fitting: (


Best combo    : (256, 128, 64)/relu/Regularization=0.001
Architecture  : (256, 128, 64)
Activation    : relu
Alpha (L2)    : 0.001
Solver        : adam
Train error   : 0.0105
Test error    : 0.2916


In [21]:

mlp_best = MLPClassifier(
    hidden_layer_sizes=best_arch,
    activation=best_act,
    solver=best_solver,
    alpha=best_regularizer,
    random_state=42
)
mlp_best.fit(X_train_scaled, y_train)
mlp_predictions = mlp_best.predict(X_test_scaled)

cm_mlp = confusion_matrix(y_test, mlp_predictions)
acc_mlp = accuracy_score(y_test, mlp_predictions)
report_mlp = classification_report(y_test, mlp_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_mlp, fp_mlp, fn_mlp, tp_mlp = cm_mlp.ravel()

print(f"Neural Network ({best_mlp_label}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_mlp['Non-Remote']['precision']:>12.4f} {report_mlp['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_mlp['Non-Remote']['recall']:>12.4f} {report_mlp['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_mlp['Non-Remote']['f1-score']:>12.4f} {report_mlp['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_mlp['Non-Remote']['support']:>12} {report_mlp['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_mlp:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_mlp['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_mlp['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_mlp,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_mlp), str(fp_mlp)], [str(fn_mlp), str(tp_mlp)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Neural Network ({best_mlp_label}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Neural Network ((256, 128, 64)/relu/Regularization=0.001) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.7782       0.5300
Recall                          0.8088       0.4834
F1 Score                        0.7932       0.5056
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.7084
Macro Avg F1                    0.6494
Weighted Avg F1                 0.7045
